In [0]:
spark.sql("USE workspace.steam")
before = spark.table("silver_steam").count()
dates = [str(r.load_date) for r in spark.sql("""
    SELECT DISTINCT load_date FROM bronze_steam
    WHERE load_date >= DATE'2026-10-09'
    ORDER BY load_date
""").collect()]
print("silver rows before:", before)
print("dates to replay:", dates)

In [0]:
spark.sql("DELETE FROM silver_steam")
print("silver emptied")

for d in dates:
    dbutils.notebook.run("./03_bronze_to_silver", 3600, {"date": d, "batch_id": ""})
    print(d, "done | silver rows now:", spark.table("silver_steam").count())

In [0]:
display(spark.sql("""
    SELECT COUNT(*)                         AS silver_rows,
           COUNT(DISTINCT appid)            AS distinct_appids,
           COUNT(average_forever_min)       AS playtime_filled,
           COUNT(userscore)                 AS userscore_filled,
           COUNT(score_rank)                AS score_rank_filled
    FROM silver_steam
"""))